# 03 - Diagnostic Test

Quick 10-step training test to diagnose speed issues.

**Purpose:** Measure time for each component and identify bottlenecks.

**Output:** Detailed timing log saved to results/diagnostic_log.txt

In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
import os
import time
import json
import sys

os.chdir("/kaggle/working")
if not os.path.exists("fingpt-qlora"):
    !git clone https://github.com/W-Kaski/fingpt-qlora.git
os.chdir("fingpt-qlora")

log_path = "results/diagnostic_log.txt"
os.makedirs("results", exist_ok=True)

class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()

log_file = open(log_path, "w")
sys.stdout = Tee(sys.__stdout__, log_file)
sys.stderr = Tee(sys.__stderr__, log_file)

def log(msg):
    timestamp = time.strftime("%H:%M:%S")
    print(f"[{timestamp}] {msg}")

log("Diagnostic test started")

In [ ]:
import torch

log("=== GPU Check ===")
log(f"GPU: {torch.cuda.get_device_name(0)}")
log(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
log(f"PyTorch: {torch.__version__}")

In [ ]:
log("=== Data Pipeline ===")
t0 = time.time()

if not os.path.exists("data/splits/train.json"):
    log("Running data pipeline...")
    !python -m src.data.download
    !python -m src.data.preprocess
    !python -m src.data.format_chat
    !python -m src.data.merge_datasets
    !python -m src.data.splits

t1 = time.time()
log(f"Data pipeline time: {t1-t0:.1f}s")

with open("data/splits/train.json") as f:
    train_data = json.load(f)
with open("data/splits/val.json") as f:
    val_data = json.load(f)

log(f"Train examples: {len(train_data)}")
log(f"Val examples: {len(val_data)}")

In [ ]:
log("=== Model Loading ===")
t0 = time.time()

from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

t1 = time.time()
log(f"Model loading time: {t1-t0:.1f}s")
log(f"VRAM after loading: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
log("=== LoRA Setup ===")
t0 = time.time()

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

t1 = time.time()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
log(f"LoRA setup time: {t1-t0:.1f}s")
log(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
log("=== Dataset Preparation ===")
t0 = time.time()

from datasets import Dataset

def load_split(path):
    with open(path) as f:
        data = json.load(f)
    texts = []
    for ex in data:
        messages = []
        for turn in ex["conversations"]:
            role = "assistant" if turn["from"] == "gpt" else turn["from"]
            messages.append({"role": role, "content": turn["value"]})
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append({"text": text})
    return Dataset.from_list(texts)

train_dataset = load_split("data/splits/train.json")
val_dataset = load_split("data/splits/val.json")

t1 = time.time()
log(f"Dataset prep time: {t1-t0:.1f}s")
log(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# Check token lengths
lengths = [tokenizer(train_dataset[i]["text"], return_tensors="pt")["input_ids"].shape[1] for i in range(min(50, len(train_dataset)))]
log(f"Token lengths (50 samples): min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)//len(lengths)}")
log(f"Max seq length: {MAX_SEQ_LENGTH}")

In [ ]:
log("=== Training Speed Test (10 steps) ===")

from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="outputs/diagnostic",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    max_steps=10,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    max_seq_length=MAX_SEQ_LENGTH,
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    eval_strategy="no",
    seed=42,
    report_to="none",
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_dataset, args=training_args)

log("Starting 10-step training...")
t0 = time.time()
trainer.train()
t1 = time.time()

total_time = t1 - t0
time_per_step = total_time / 10

log(f"10 steps completed in {total_time:.1f}s")
log(f"Time per step: {time_per_step:.2f}s")
log(f"Expected (Unsloth): 1-2s per step")
log(f"Slowdown: {time_per_step / 1.5:.1f}x")

for entry in trainer.state.log_history:
    if "loss" in entry:
        log(f"Step {entry['step']}: loss={entry['loss']:.4f}")

In [ ]:
log("=" * 60)
log("DIAGNOSTIC SUMMARY")
log("=" * 60)
log(f"Training speed: {time_per_step:.2f}s per step")
log(f"Possible causes:")
if time_per_step > 5:
    log("  - Long sequences, try reducing MAX_SEQ_LENGTH")
if max(lengths) > 1500:
    log(f"  - Max token length {max(lengths)} is very long")
log("Recommendations:")
log("  1. Enable packing=True")
log("  2. Reduce max_seq_length to 1024 or 512")
log(f"\\nLog saved to: {log_path}")
log("Download from Output tab after kernel completes.")